# ***Predicting Firm-Level Loss Behaviour and Corporate Income Tax (CIT) Revenue Risk in Kenya***


### Authors

Brian Kahiu, John Karanja, Cyrus Mutuku, Catherine Gachiri, Fredrick Nzeve, Grace Kinyanjui, Jeremy Onsongo


### 1.0 Business Understanding

### Background Information

The Kenya Revenue Authority (KRA), established in 1995, is responsible for collecting all national government revenue. Corporate Income Tax (CIT) is levied at 30% for resident companies and non-residents under the Income Tax Act, with sector-specific incentives available through Special Economic Zones (SEZs), Capital Deductions and Export Processing Zones (EPZs).

Companies must file annual CIT returns (ITC2 form) electronically via the iTax platform within six months of their accounting year-end, supported by valid electronic Tax Invoice Management System (eTIMS) invoices. Taxable income is calculated as gross income less allowable business expenses. A company is considered tax resident if incorporated in Kenya or if management and control is exercised locally.

### Business Problem Definition

Kenya has persistently failed to meet Corporate Income Tax (CIT) revenue targets. The high prevalence of firms reporting losses significantly erodes the effective tax base, creating fiscal deficit. The central problem is the lack of an empirical, data-driven framework for:

1. Identifying which firm-level characteristics are associated with loss reporting.
2. Proactively identifying high-risk firms and sectors.
3. Assessing how firm-level loss behavior translates into systemic CIT revenue risk.

### Our Solution

An automated risk scoring system that:

1. Processes firm-level CIT return data using the methodology outlined in the project proposal.
2. Employs an iterative modeling approach, beginning with interpretable logistic regression as a primary benchmark.
3. Applies machine learning to identify high-risk loss-reporting firms for targeted compliance.

### Project Objectives

***General Objective***

To predict the probability of a firm reporting a loss

***Specific Objectives***

1. To empirically identify firm-level characteristics associated with loss reporting in CIT returns.
2. To develop a supervised predictive model estimating the probability of a firm reporting a loss.
3. To assess the concentration and distribution of loss behavior across sectors and firm groups.
4. To translate firm-level loss probabilities into insights on aggregate CIT revenue risk.

### Methodology

This project follows the Cross-Industry Standard Process for Data Mining (CRISP-DM) to ensure a structured, transparent, and policy-relevant analytics workflow.

### Business Understanding

Stakeholder needs were identified, the business problem was defined, and success metrics were established to align analytical outputs with compliance and fiscal objectives.

### Data Understanding

Corporate Income Tax return data for 2024 were explored to assess structure and data quality.


### Primary Stakeholders

1. KRA Compliance Directors

***Problem:*** Manual audit selection misses high-risk loss-reporting firms

***Need:*** Prioritize firms with highest evasion probability for investigation

***Business Value:*** Improved audit efficiency and revenue recovery

2. Tax Policy Analysts at National Treasury

***Problem:*** Revenue forecasting uncertainty due to loss declaration patterns

***Need:*** Data-driven risk assessment for fiscal planning and budgeting

***Business Value:*** Improved accuracy in CIT revenue projections

3. Field Tax Officers

***Problem:*** Wasted time on low-risk audits with minimal revenue recovery

***Need:*** Focus investigations on firms with highest probability of tax avoidance

***Business Value:*** Higher productivity and improved targeting outcomes

### 2.0 Data Description
The analysis uses 2024 Corporate Income Tax return data containing 313,870 firm-year observations across 61 variables (47 numeric, 14 categorical). The dataset includes financial data, sector classifications, and firm characteristics from administrative filings.

### 3.0 Modeling Scope Definition

Validity: Retain only active businesses with positive turnover

Target: Flag firms as "Risk" (is_loss = 1) if Profit Before Tax is negative

Sector Standardization: Clean sector names and consolidate rare sectors




### 1.0 Import the necessary libraries

We import the key libraries

In [1]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import xgboost as xgb

# Machine learning Preprocessing & Utilities
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine learning Metrics
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss, log_loss
)

# Model interpretability
import shap

# Model saving
import joblib

# System utilities
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


### 2.0 Data loading and Initial Checks
Here, we imported the raw tax data and examined its basic structure.

In [2]:
df = pd.read_csv("CIT2024.csv", low_memory=False)
df.shape

(313870, 61)

In [3]:
df.info()
print(df.dtypes.value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313870 entries, 0 to 313869
Data columns (total 61 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   unique_id                      313870 non-null  float64
 1   business_type                  313870 non-null  object 
 2   business_subtype               310874 non-null  object 
 3   epz_effective_dt               149 non-null     object 
 4   period_from                    313870 non-null  object 
 5   period_to                      313870 non-null  object 
 6   filing_date                    313870 non-null  object 
 7   is_nil_return                  313870 non-null  object 
 8   return_type                    313870 non-null  object 
 9   assmt_type                     313870 non-null  object 
 10  eff_dt_com_activity            85 non-null      object 
 11  sector                         313862 non-null  object 
 12  division_                     

In [4]:
df.head(5)

,unique_id,business_type,business_subtype,epz_effective_dt,period_from,period_to,filing_date,is_nil_return,return_type,assmt_type,...,prof_loss_tax_div_bal_st,empexp__salary_wages,init_plant_mach_allow,init_indu_buld_allow,cap_allw_indu_buld,wear_tear_dedc_rbm,wear_tear_dedc_slm,deduct_agri_land,tot_allow_deductions,avg_no_of_employees
0,1.210000e+09,Company,Private Company,NaN,1/1/2024,31/12/2024,27/06/2025,N,Original,S,...,8191.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,1.210001e+10,Company,Private Company,NaN,1/1/2024,31/12/2024,27/06/2025,Y,Original,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,1.210002e+10,Company,Private Company,NaN,1/1/2024,31/12/2024,27/05/2025,N,Original,S,...,151384.13,250000.0,0.0,0.0,0.0,1853.1,0.0,0.0,1853.1,NaN
3,1.210002e+10,Company,Private Company,NaN,1/1/2024,31/12/2024,20/05/2025,Y,Original,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.210002e+10,Company,Private Company,NaN,1/1/2024,31/12/2024,29/06/2025,N,Original,S,...,114704.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


### 2.1 Missing Values and Duplicate Check

- We checked for missing values and duplicates 

In [5]:
# 1. Duplicate check
print("Duplicated Rows",df.duplicated().sum())

# 2. Missing values check
df.isnull().mean().mul(100).round(3).sort_values(ascending=False)
print("Top 10 columns with missing values in percentage (%)")
print(df.isnull().mean().mul(100).round(3).sort_values(ascending=False).head(10))


Duplicated Rows 3011
Top 10 columns with missing values in percentage (%)
eff_dt_com_activity             99.973
epz_effective_dt                99.953
income_tax_exp                  96.193
avg_no_of_employees             80.814
class_                          65.182
prof_loss_tax_div_bal_st        64.151
insurance_comp                  64.151
oi_dividend                     64.151
oi_commision                    64.151
oi_natural_resource_payments    64.151
dtype: float64


The analysis revealed that missing values were concentrated in a small subset of columns, while most variables maintained high completeness.

### 2.2 Initial Data Cleaning Actions
